<p align="center"><b>Assignment 11</b></p>




Tiny ImageNet Dataset (64 x 64 pixel RGB images)

* Dataset Link: Tiny ImageNet on Hugging Face / Kaggle (or direct Colab download via [http://cs231n.stanford.edu/tiny-imagenet200.zip](http://cs231n.stanford.edu/tiny-imagenet-200.zip))

* Setup: Select any 3 classes with 200 images each (Total = 600 images of size 64x64x3).

 Tasks
 1. CNN Basics on a 3×3 Matrix:

    Write a program to initialize a random 3×3 matrix and implement:
  * Convolution, Padding (P=1), and Activation Functions (ReLU/Sigmoid).  Max Pooling, Min Pooling, and Global Average Pooling.
  * Flattening, Fully Connected (Dense) layers, Batch Normalization, Dropout, Early Stopping.
 2. Custom CNN & Overfitting Analysis (600 Images):

    Write a program to build and train a custom CNN using the components from Task 1 on the 600-image dataset. Use 70% /15% /15% Split for model training. Plot training and validation loss curves to test for underfitting and overfitting.
    
 3. Transfer Learning Comparison (600 Images):

    Write a program to implement VGG16, MobileNet, and ResNet on the same 600image dataset and compare their test accuracy, precision, recall and F1 score against your custom CNN.

In [ ]:
!pip install tensorflow opencv-python numpy matplotlib scikit-learn pandas

In [ ]:
import os

zip_path = "/content/tiny-imagenet-200.zip"
extract_path = "/content/data"

if not os.path.exists(extract_path):
    !unzip -q "$zip_path" -d "$extract_path"

print("Dataset ready!")

In [ ]:
import os
import zipfile

zip_path = "/content/tiny-imagenet-200.zip"

print("File exists:", os.path.exists(zip_path))

if os.path.exists(zip_path):
    print("File size:", os.path.getsize(zip_path) / (1024 * 1024), "MB")
    print("Is valid ZIP:", zipfile.is_zipfile(zip_path))

    with open(zip_path, "rb") as f:
        print("File signature:", f.read(20))

In [ ]:
import cv2
import numpy as np

X = []
y = []

classes = sorted(os.listdir(train_path))[:3]

for label, cls in enumerate(classes):
    folder = os.path.join(train_path, cls, "images")

    count = 0

    for file in os.listdir(folder):
        if count == 200:
            break

        img = cv2.imread(os.path.join(folder, file))

        if img is not None:
            img = cv2.resize(img, (64, 64))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            X.append(img)
            y.append(label)
            count += 1

X = np.array(X, dtype="float32") / 255.0
y = np.array(y)

print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(len(X_train), len(X_val), len(X_test))

In [ ]:
import numpy as np

a = np.random.rand(3, 3)
k = np.ones((3, 3))

p = np.pad(a, 1)
conv = np.zeros((3, 3))

for i in range(3):
    for j in range(3):
        conv[i,j] = np.sum(p[i:i+3, j:j+3] * k)

print("Convolution:\n", conv)
print("ReLU:\n", np.maximum(0, conv))
print("Sigmoid:\n", 1/(1+np.exp(-conv)))

print("Max Pooling:", np.max(conv[:2,:2]))
print("Min Pooling:", np.min(conv[:2,:2]))
print("Global Average Pooling:", np.mean(conv))
print("Flatten:", conv.flatten())

In [ ]:
import numpy as np

a = np.random.rand(3, 3)
k = np.ones((3, 3))

p = np.pad(a, 1)
conv = np.zeros((3, 3))

for i in range(3):
    for j in range(3):
        conv[i,j] = np.sum(p[i:i+3, j:j+3] * k)

print("Convolution:\n", conv)
print("ReLU:\n", np.maximum(0, conv))
print("Sigmoid:\n", 1/(1+np.exp(-conv)))

print("Max Pooling:", np.max(conv[:2,:2]))
print("Min Pooling:", np.min(conv[:2,:2]))
print("Global Average Pooling:", np.mean(conv))
print("Flatten:", conv.flatten())

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout

x = tf.random.normal((1, 9))

print("Dense:", Dense(4)(x).numpy())
print("Batch Normalization:", BatchNormalization()(x, training=True).numpy())
print("Dropout:", Dropout(0.5)(x, training=True).numpy())

early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3
)

print("Early Stopping created")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([
    tf.keras.Input(shape=(64,64,3)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
early = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early]
)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Custom CNN Accuracy:", acc)

In [ ]:
from tensorflow.keras.applications import VGG16, MobileNetV2, ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
def train_model(base_model):

    base = base_model(
        weights='imagenet',
        include_top=False,
        input_shape=(64,64,3)
    )

    base.trainable = False

    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(3, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=5,
              validation_data=(X_val,y_val), verbose=1)

    pred = np.argmax(model.predict(X_test), axis=1)

    return [
        accuracy_score(y_test,pred),
        precision_score(y_test,pred,average='weighted',zero_division=0),
        recall_score(y_test,pred,average='weighted',zero_division=0),
        f1_score(y_test,pred,average='weighted',zero_division=0)
    ]

In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobile_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre

def run_model(base_model, preprocess):
    base = base_model(weights='imagenet', include_top=False,
                      input_shape=(64,64,3))
    base.trainable = False

    inp = tf.keras.Input(shape=(64,64,3))
    x = tf.keras.layers.Lambda(preprocess)(inp)
    x = base(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(64, activation='relu')(x)
    out = Dense(3, activation='softmax')(x)

    m = tf.keras.Model(inp, out)
    m.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

    m.fit(X_train*255, y_train, epochs=5,
          validation_data=(X_val*255,y_val), verbose=1)

    pred = np.argmax(m.predict(X_test*255), axis=1)

    return [
        accuracy_score(y_test,pred),
        precision_score(y_test,pred,average='weighted',zero_division=0),
        recall_score(y_test,pred,average='weighted',zero_division=0),
        f1_score(y_test,pred,average='weighted',zero_division=0)
    ]

vgg = run_model(VGG16, vgg_pre)
mobile = run_model(MobileNetV2, mobile_pre)
resnet = run_model(ResNet50, resnet_pre)

In [ ]:
import pandas as pd

custom_pred = np.argmax(model.predict(X_test), axis=1)

custom = [
    accuracy_score(y_test,custom_pred),
    precision_score(y_test,custom_pred,average='weighted',zero_division=0),
    recall_score(y_test,custom_pred,average='weighted',zero_division=0),
    f1_score(y_test,custom_pred,average='weighted',zero_division=0)
]

df = pd.DataFrame({
    'Model':['Custom CNN','VGG16','MobileNetV2','ResNet50'],
    'Accuracy':[custom[0],vgg[0],mobile[0],resnet[0]],
    'Precision':[custom[1],vgg[1],mobile[1],resnet[1]],
    'Recall':[custom[2],vgg[2],mobile[2],resnet[2]],
    'F1 Score':[custom[3],vgg[3],mobile[3],resnet[3]]
})

print(df)